# Lab 2: GenAI-Augmented Visual Analytics

**Author:** Jason A. Minton, Ph.D.
**Purpose:** A hands-on companion to the RAG lab. Where the first lab demonstrated grounded question-answering over a document, this one demonstrates LLM augmentation of the *data visualization lifecycle* — describing a dataset, flagging data-quality concerns, turning natural-language questions into charts, and narrating what those charts show.

---

## What you will build

A working Streamlit application that lets a user upload a CSV (or use a synthetic Washington water-monitoring sample) and then:

1. Generates a plain-English description of the dataset
2. Flags data-quality concerns *before* any analysis takes place
3. Accepts plain-English questions and turns them into Plotly charts via the LLM's tool-use API
4. Writes a short narrative caption for each chart explaining what it shows

By the end you will have a notebook that walks through each concept individually, an assembled `app.py` you can run locally, and instructions for deploying the app to Streamlit Community Cloud — giving you a live URL you can share with anyone.

## Why this matters

Two reasons that go beyond "Streamlit and Plotly are popular libraries."

**First, this lab puts a research finding into operational form.** The doctoral research this lab draws on (Minton, 2025) finds that Enterprise Architects and Technology Architects often lack awareness of whether their organizations are measuring data quality, with gaps in how data-quality frameworks and specifications are integrated into Enterprise Architecture practice. The data-quality flagging step in this lab is the *operationalization* of that finding: a working tool that makes data-quality measurement visible at the moment of analysis rather than leaving it as an unexamined assumption. Talburt's definition — *data quality is conformance to data specifications* — anchors the LLM prompt that drives this step, which means the research's foundational definition is doing real work in production code.

**Second, it converts reading into practice.** Integration of LLMs and generative models into the data visualization lifecycle — data preparation, natural-language chart generation, AI-narrated dashboards, and interactive analytical reasoning — is easy to read about and different to build. This lab exercises every one of those patterns with widely used libraries (pandas, plotly). After building it, you can speak to each pattern from direct implementation rather than from study.

## How to use this lab

Work the cells in order. Each part introduces a concept, lets you exercise it on the sample dataset, and pauses for a brief reflection. Then in Part 6 you'll see how the same functions get composed into the final Streamlit app.

This notebook is designed to run in **Google Colab** (free, no setup needed), though it also runs in local Jupyter. The final assembled app runs in Streamlit, which is a separate environment — see the README for local-run and deployment instructions.

---

## Setup

Three things to put in place before the first code cell.

**1. Anthropic API key.** Same as the RAG lab. Sign up at [console.anthropic.com](https://console.anthropic.com), generate a key, and either store it in Colab secrets as `ANTHROPIC_API_KEY` or set it in your local environment. This lab uses well under a dollar of API credit even with significant experimentation.

**2. Sample data file.** This lab is self-contained — the sample data generator is included inline below. You do not need to upload anything to run the notebook end-to-end. If you want to try the app with your own CSV later, save the assembled `app.py` and use it from Streamlit; the notebook focuses on the synthetic environmental dataset because it has deliberate data-quality issues that make the flagging step demonstrate clearly.

**3. Python libraries.** The first code cell installs them. About 30 seconds the first time, instant after that.

In [1]:
from transformers.models.instructblip.modeling_instructblip import InstructBlipQFormerAttention
# Install the libraries this lab uses.
!pip install --quiet anthropic pandas plotly

print("Libraries installed.")

/Users/jaminton/Downloads/repo/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Libraries installed.


In [2]:
# Load the Anthropic API key from Colab secrets (or fall back to environment).
import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("API key loaded from Colab secrets.")
except ImportError:
    if 'ANTHROPIC_API_KEY' not in os.environ:
        raise RuntimeError("Set ANTHROPIC_API_KEY in your environment before continuing.")
    print("API key found in environment.")

API key found in environment.


### Generate the synthetic sample dataset

The cell below generates 121 rows of synthetic Washington water-monitoring data across five sites and twenty-four weeks. The data has deliberate data-quality issues seeded in — missing values, impossible pH readings, an outlier temperature, a duplicate row, and a few cells with the wrong type. These exist so the data-quality flagging step has something concrete to surface.

Reading the seeding code below is itself instructive: you can see exactly what kinds of failures the flagging step is supposed to catch.

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


def generate_sample_dataset(seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sites = [
        {"site_id": "WAEC-001", "site_name": "Spokane River at Riverside",      "basin": "Spokane"},
        {"site_id": "WAEC-002", "site_name": "Lake Washington at Madison Park", "basin": "Puget Sound"},
        {"site_id": "WAEC-003", "site_name": "Yakima River at Cle Elum",        "basin": "Yakima"},
        {"site_id": "WAEC-004", "site_name": "Columbia River at Wenatchee",     "basin": "Columbia"},
        {"site_id": "WAEC-005", "site_name": "Chehalis River at Centralia",     "basin": "Chehalis"},
    ]
    start = datetime(2025, 1, 1)
    records = []
    for site in sites:
        for week in range(24):
            date = start + timedelta(weeks=week, days=int(rng.integers(0, 3)))
            base_temp = 45 + 20 * np.sin(week / 24 * np.pi)
            records.append({
                "site_id":           site["site_id"],
                "site_name":         site["site_name"],
                "basin":             site["basin"],
                "sample_date":       date.strftime("%Y-%m-%d"),
                "temperature_f":     round(float(base_temp + rng.normal(0, 3)), 1),
                "ph":                round(float(7.2 + rng.normal(0, 0.3)), 2),
                "dissolved_oxygen_mgL": round(float(9.5 + rng.normal(0, 1.2)), 2),
                "turbidity_ntu":     round(float(abs(rng.normal(3.0, 1.5))), 2),
                "conductivity_uScm": round(float(180 + rng.normal(0, 40)), 1),
            })
    df = pd.DataFrame(records)
    # Seed data-quality issues
    df.loc[7,   "temperature_f"]        = np.nan
    df.loc[23,  "temperature_f"]        = np.nan
    df.loc[44,  "dissolved_oxygen_mgL"] = np.nan
    df.loc[88,  "dissolved_oxygen_mgL"] = np.nan
    df.loc[101, "ph"]                   = np.nan
    df.loc[15,  "ph"]                   = 25.4   # impossible pH
    df.loc[62,  "temperature_f"]        = 142.0  # impossible temperature
    df.loc[33,  "turbidity_ntu"]        = 4500.0 # wrong scale
    df["ph"] = df["ph"].astype(object)
    df.loc[50, "ph"] = "7.2"   # type inconsistency: string in a numeric column
    df.loc[78, "ph"] = "7.5"
    duplicate = df.iloc[10].copy()
    df = pd.concat([df, duplicate.to_frame().T], ignore_index=True)
    return df


df = generate_sample_dataset()
print(f"Generated dataset: {len(df)} rows × {len(df.columns)} columns")
df.head()

Generated dataset: 121 rows × 9 columns


,site_id,site_name,basin,sample_date,temperature_f,ph,dissolved_oxygen_mgL,turbidity_ntu,conductivity_uScm
0,WAEC-001,Spokane River at Riverside,Spokane,2025-01-01,41.9,7.43,10.63,0.07,127.9
1,WAEC-001,Spokane River at Riverside,Spokane,2025-01-10,48.0,7.11,9.48,1.72,215.2
2,WAEC-001,Spokane River at Riverside,Spokane,2025-01-15,50.4,7.54,10.06,1.71,194.8
3,WAEC-001,Spokane River at Riverside,Spokane,2025-01-24,49.8,7.46,9.44,2.72,152.8
4,WAEC-001,Spokane River at Riverside,Spokane,2025-01-29,54.5,7.07,9.08,3.8,194.6


---

# Part 1 — Loading and Inspecting Data with pandas

## Why this is where serious data work begins

Before any LLM ever sees this dataset, you need to know what you're holding. Schema, types, missingness, duplicates, distribution — these are the things a careful analyst examines first. They're also exactly the things an LLM needs to know about in order to produce useful output downstream. If the LLM doesn't know that `ph` contains some string values masquerading as numbers, it cannot warn the user about it.

**Analogy:** think of this step as taking inventory at the loading dock. The truck has arrived. Before anything goes onto the shelves, you count the boxes, check what was actually delivered against what was ordered, and note anything that looks damaged. The LLM is the warehouse staff who will help organize and use the inventory — but it cannot do its job well if the inventory itself was never examined.

pandas — the de facto Python library for tabular data — is the right tool for this stage. The methods you'll use most often are `.head()`, `.info()`, `.describe()`, `.isna().sum()`, `.duplicated().sum()`, and `.dtypes`. None of them is glamorous; together they are the foundation everything else rests on.

In [4]:
# Schema and types — what columns do we have, and what type is each?
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   site_id               121 non-null    object
 1   site_name             121 non-null    object
 2   basin                 121 non-null    object
 3   sample_date           121 non-null    object
 4   temperature_f         119 non-null    object
 5   ph                    120 non-null    object
 6   dissolved_oxygen_mgL  119 non-null    object
 7   turbidity_ntu         121 non-null    object
 8   conductivity_uScm     121 non-null    object
dtypes: object(9)
memory usage: 8.6+ KB


In [5]:
# Descriptive statistics for numeric columns.
# Look at the mean, min, and max — anything that looks impossible?
df.describe(include='all')

,site_id,site_name,basin,sample_date,temperature_f,ph,dissolved_oxygen_mgL,turbidity_ntu,conductivity_uScm
count,121,121,121,121,119.0,120.00,119.00,121.00,121.0
unique,5,5,5,63,93.0,69.00,98.00,109.00,112.0
top,WAEC-001,Spokane River at Riverside,Spokane,2025-03-13,63.5,7.11,10.15,1.72,147.0
freq,25,25,25,5,4.0,6.00,3.00,2.00,2.0


In [6]:
# Missing-value counts per column.
# Completeness is the most common data-quality dimension to examine.
print("Missing values per column:")
print(df.isna().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

Missing values per column:
site_id                 0
site_name               0
basin                   0
sample_date             0
temperature_f           2
ph                      1
dissolved_oxygen_mgL    2
turbidity_ntu           0
conductivity_uScm       0
dtype: int64

Duplicate rows: 1


### Reflection 1 — What you can already see, before any LLM is involved

Look at the three cells above. With pandas alone — no LLM, no AI, no clever software — you can already see most of what's wrong with this dataset:

1. **The `ph` column is type `object`.** That's pandas' way of telling you "this column contains mixed types — probably strings mixed with numbers." For a column that should be purely numeric, this is a structural data-quality problem.

2. **The `temperature_f` column has a maximum of 142.0.** Surface water in Washington State does not reach 142°F. Whatever produced that reading was either a sensor malfunction, a unit confusion, or a data-entry error.

3. **The `ph` column has a maximum of 25.4.** The pH scale runs from 0 to 14. Anything above that is not pH; it's something else entered into the pH column.

4. **There is one duplicate row.** Two rows in this dataset are identical — same site, same date, same readings — which almost certainly means the same observation was entered twice rather than being two independent measurements.

5. **There are missing values** in three columns: temperature, dissolved oxygen, and pH.

Now here is the question that ties this to the underlying research: how many organizations actually look at these things before feeding data to a downstream system? Minton (2025) finds that Enterprise Architects often lack awareness of whether their own organizations are measuring data quality. The cells above show what it looks like when measurement is done — when an analyst takes the four minutes required to examine completeness, distribution, and type before any analysis runs. The whole rest of this lab is about building a tool that does this examination automatically, so that awareness gap gets closed by design rather than by individual vigilance.

---

# Part 2 — LLM-Generated Dataset Descriptions

## The first useful pattern: hand the LLM a schema, get back prose

The simplest useful thing an LLM can do for a data-visualization pipeline is *describe what a dataset appears to be*. Given the column names, types, and a few sample rows, a competent LLM can produce a one-paragraph plain-English description that helps a user orient themselves to data they've never seen before.

**Why this matters:** real organizational datasets often arrive with vague names, missing documentation, and no data dictionary. A short LLM-generated description is not a substitute for proper data documentation, but it is a useful first orientation — particularly for a non-technical stakeholder being asked to make decisions from the data.

**The pattern:** we build a compact schema summary (columns, types, sample values), feed it to the LLM, and ask for a short, restrained description. The key word in the prompt is *restrained* — we do not want the LLM to speculate beyond what the schema and sample support. This is the same principle that drove the grounded-answer prompting in the RAG lab.

In [7]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"


def schema_summary(df):
    '''A compact schema description suitable for inclusion in a prompt.'''
    lines = [f"Dataset has {len(df)} rows and {len(df.columns)} columns."]
    for col in df.columns:
        sample = df[col].dropna().head(3).tolist()
        lines.append(f"  - {col} ({df[col].dtype}): example values {sample}")
    return "\n".join(lines)


schema_text = schema_summary(df)
sample_csv = df.head(5).to_csv(index=False)

print(schema_text)

Dataset has 121 rows and 9 columns.
  - site_id (object): example values ['WAEC-001', 'WAEC-001', 'WAEC-001']
  - site_name (object): example values ['Spokane River at Riverside', 'Spokane River at Riverside', 'Spokane River at Riverside']
  - basin (object): example values ['Spokane', 'Spokane', 'Spokane']
  - sample_date (object): example values ['2025-01-01', '2025-01-10', '2025-01-15']
  - temperature_f (object): example values [41.9, 48.0, 50.4]
  - ph (object): example values [7.43, 7.11, 7.54]
  - dissolved_oxygen_mgL (object): example values [10.63, 9.48, 10.06]
  - turbidity_ntu (object): example values [0.07, 1.72, 1.71]
  - conductivity_uScm (object): example values [127.9, 215.2, 194.8]


In [8]:
# Ask the LLM to describe the dataset.
response = client.messages.create(
    model=MODEL,
    max_tokens=400,
    messages=[{
        "role": "user",
        "content": (
            "Here is a dataset schema and a small sample.\n\n"
            f"{schema_text}\n\n"
            f"Sample (CSV):\n{sample_csv}\n\n"
            "Write one short paragraph (3-4 sentences) describing what this dataset "
            "appears to contain. Be specific about what is being measured and any "
            "structure you can infer. Do not speculate beyond what the schema and "
            "sample support."
        ),
    }],
)

print(response.content[0].text)

This dataset contains water quality measurements from monitoring sites across different river basins in Washington state. Each row represents a sample collection event at a specific site on a given date, recording five key water quality parameters: temperature (in Fahrenheit), pH level, dissolved oxygen concentration (mg/L), turbidity (NTU), and electrical conductivity (μS/cm). The sample shown includes multiple temporal measurements from a single monitoring location (Spokane River at Riverside) taken at different dates in January 2025, suggesting the dataset tracks water quality changes over time across multiple sites.


### Reflection 2 — On restraint as a virtue

Read the LLM's description. A few things worth noticing:

1. **Did the LLM speculate beyond what the data supports?** It might say "this appears to be water-quality monitoring data from Washington State" — which is *speculation* in a strict sense (the schema doesn't say "Washington" anywhere), but it's the kind of inference a human analyst would also make from the site names. The harder question is whether the LLM made any claims that go beyond what a reasonable inference would support — claims about purpose, regulatory context, organizational ownership, or downstream use. Those would be problematic.

2. **The prompt's explicit instruction not to speculate matters.** If you remove that instruction and try again, you'll often get a longer description with more confident claims about purpose and intent. The instruction is doing real work.

3. **Design note:** the design choice here is *restraint over flair*. The LLM is excellent at writing fluent, confident prose. That is exactly the failure mode you want to engineer against in a data-quality context. The prompt's specific request — *be specific about what is being measured*, *do not speculate beyond what the schema and sample support* — is what shapes the model's behavior toward the kind of careful description that you would trust in a regulatory or scientific setting.

---

# Part 3 — Data-Quality Flagging (The Research Moment)

## The single most important step in this lab

This is the step where the underlying research recommendation becomes operational. Minton (2025) argues that Enterprise Architects often lack awareness of whether their organizations are measuring data quality. This step *measures* data quality — automatically, at the moment of ingestion, before any analysis happens — and surfaces the results for the user to see.

**The pattern:** we feed the LLM both the schema and a quantitative profile of the dataset (missing-value counts, descriptive statistics, duplicate counts), and we ask it to identify data-quality concerns. The system prompt explicitly anchors the work to Talburt's (2015) definition — *data quality is conformance to data specifications* — and asks for structured JSON output that we can render with appropriate visual severity in the final app.

**Why this is the research moment:** the dissertation's foundational definition of data quality — Talburt's — is the one we're using to anchor the LLM. The framework the research calls for — deliberate, explicit measurement of data quality as an integrated part of how organizations work with data — is what this step makes operational.

**A note on framework grounding:** Wang and Strong (1996) identified fifteen data-quality dimensions across four categories (intrinsic, contextual, representational, accessibility). DAMA-DMBOK and ISO 8000 organize similar concerns. Talburt's definition unifies them under a conformance-to-specifications framing. The prompt below references Talburt directly because that is the grounding the dissertation uses; you could substitute or extend with Wang & Strong or DAMA if you wanted to make different framework allegiances explicit.

In [9]:
import json

# Build a compact profile of the data quality dimensions worth reporting.
profile_text = (
    f"Missing counts per column:\n{df.isna().sum().to_string()}\n\n"
    f"Numeric column statistics:\n{df.describe(include='all').to_string()}\n\n"
    f"Duplicate row count: {int(df.duplicated().sum())}"
)

response = client.messages.create(
    model=MODEL,
    max_tokens=800,
    system=(
        "You are a data quality reviewer. Talburt (2015) defines data quality as "
        "conformance to data specifications. Your job is to surface concerns that a "
        "downstream analyst or AI system should know about before relying on this data. "
        "Focus on: missing values, impossible or out-of-range values, type "
        "inconsistencies, duplicates, and anomalous distributions. Output a JSON list "
        "of concerns. Each concern is an object with keys 'severity' (one of: high, "
        "medium, low) and 'description' (one sentence). Output ONLY the JSON list, "
        "nothing else."
    ),
    messages=[{
        "role": "user",
        "content": (
            f"Schema:\n{schema_text}\n\n"
            f"Profile (descriptive statistics and counts):\n{profile_text}\n\n"
            "List the data quality concerns."
        ),
    }],
)

raw = response.content[0].text.strip()
if raw.startswith("```"):
    raw = raw.split("```", 2)[1]
    if raw.startswith("json"):
        raw = raw[4:]
    raw = raw.strip().rstrip("`").strip()

concerns = json.loads(raw)

# Print them with severity formatting.
for concern in concerns:
    sev = concern.get("severity", "low").upper()
    print(f"[{sev}] {concern['description']}")

[HIGH] Numeric columns (temperature_f, ph, dissolved_oxygen_mgL, turbidity_ntu, conductivity_uScm) are stored as object/string type instead of numeric type, which will prevent proper numerical analysis.
[MEDIUM] Missing values detected in temperature_f (2 missing), dissolved_oxygen_mgL (2 missing), and ph (1 missing), which could affect water quality assessments.
[MEDIUM] One duplicate row exists in the dataset, which may represent either a data entry error or a legitimate repeated measurement that needs clarification.
[LOW] The sample_date column is stored as object/string type rather than datetime type, which may complicate time-series analysis and date-based filtering.
[LOW] Temperature value of 63.5°F appears 4 times (most frequent), which could indicate a sensor default value or calibration issue, though not impossible for water temperature.
[LOW] pH value of 7.11 appears 6 times (most frequent), which is unusually high frequency and may warrant verification for sensor accuracy or

### Reflection 3 — On what just happened, and why it matters

Read the LLM's flagged concerns. A few things worth examining carefully:

1. **Did the LLM catch the issues you saw in Part 1?** It should have flagged the impossible pH value of 25.4, the impossible temperature of 142.0, the type inconsistency in the pH column, the duplicate row, and the missing values. If it missed any, that itself is a finding worth noting — and a reason to combine LLM-based flagging with deterministic rule-based checks in production.

2. **What is the LLM doing structurally?** It is comparing the data's *actual* values and types to *implicit specifications* — knowing that pH must be in 0-14, that water temperature has physical bounds, that a numeric column should not contain strings. The "specifications" are mostly common-sense domain knowledge encoded in the model's training. For a regulatory or specialized context, you would want to pass explicit specifications into the prompt (or, better, encode them as deterministic checks and let the LLM only explain violations).

3. **The connection to the research:** the data-quality flagging step in this app is grounded in Talburt's definition of data quality as conformance to data specifications. The dissertation argued that organizations often lack awareness of whether they are measuring data quality. This step measures it automatically and shows the user the results — the practical operationalization of the research recommendation, tying technical implementation directly to scholarly argument.

---

# Part 4 — Natural-Language-to-Chart via Tool Use

## The current best-practice pattern, and why it matters

The user's question is in plain English. The output we want is a chart. There are three ways to bridge them:

1. **Let the LLM emit Python code, then exec() it.** Works in toy demos. In production it is unsafe (arbitrary code execution against your data and environment) and brittle (small syntactic errors break everything).

2. **Let the LLM emit a structured JSON spec (e.g., Vega-Lite).** Safer but still brittle — schema mismatches and malformed JSON failures happen often enough to require defensive parsing.

3. **Use the LLM's tool-use feature with a curated set of chart-creation tools.** The LLM picks exactly one tool from a fixed palette and provides arguments with a schema you defined. The Anthropic API validates the arguments and returns them as a clean Python dictionary. You then render the chart with deterministic Python code.

Option 3 is the modern best practice and the pattern this lab uses. It is safe, debuggable, and lets you put the boundary between *LLM judgment* (which chart, which columns) and *deterministic execution* (rendering the chart) exactly where it belongs.

**Analogy:** think of the LLM as a thoughtful colleague who knows the dataset but has never used your charting library. You give them a menu of available charts and the parameters each one needs. They pick from the menu and fill in the parameters. The actual chart is built by your reliable code, not by their improvisation.

**The tools we'll define:** bar chart, line chart, scatter plot, histogram, box plot — the five most common chart types for tabular data. A more sophisticated palette might include heatmaps, choropleth maps, and faceted small multiples; the principle generalizes.

In [10]:
# Define the chart tools — what the LLM can pick from.
CHART_TOOLS = [
    {
        "name": "create_bar_chart",
        "description": "Create a bar chart. Use for comparing values across categories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "x_column": {"type": "string", "description": "Categorical column for the x-axis"},
                "y_column": {"type": "string", "description": "Numeric column for the y-axis"},
                "aggregation": {
                    "type": "string",
                    "enum": ["mean", "sum", "count", "median"],
                    "description": "How to aggregate y_column when x_column has duplicate categories",
                },
                "title": {"type": "string"},
            },
            "required": ["x_column", "y_column", "aggregation", "title"],
        },
    },
    {
        "name": "create_line_chart",
        "description": "Create a line chart. Use for trends over time or continuous progressions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "x_column": {"type": "string"},
                "y_column": {"type": "string"},
                "color_column": {"type": "string", "description": "Optional grouping column; pass empty string if no grouping"},
                "title": {"type": "string"},
            },
            "required": ["x_column", "y_column", "title"],
        },
    },
    {
        "name": "create_scatter_chart",
        "description": "Create a scatter plot. Use for relationships between two numeric variables.",
        "input_schema": {
            "type": "object",
            "properties": {
                "x_column": {"type": "string"},
                "y_column": {"type": "string"},
                "color_column": {"type": "string", "description": "Optional categorical column to color by; pass empty string for none"},
                "title": {"type": "string"},
            },
            "required": ["x_column", "y_column", "title"],
        },
    },
    {
        "name": "create_histogram",
        "description": "Create a histogram. Use for understanding the distribution of a single numeric variable.",
        "input_schema": {
            "type": "object",
            "properties": {
                "column": {"type": "string"},
                "title": {"type": "string"},
            },
            "required": ["column", "title"],
        },
    },
    {
        "name": "create_box_plot",
        "description": "Create a box plot. Use for comparing distributions across categories or surfacing outliers.",
        "input_schema": {
            "type": "object",
            "properties": {
                "value_column": {"type": "string"},
                "group_column": {"type": "string", "description": "Optional categorical column to split by; pass empty string for none"},
                "title": {"type": "string"},
            },
            "required": ["value_column", "title"],
        },
    },
]

print(f"Defined {len(CHART_TOOLS)} chart tools.")

Defined 5 chart tools.


In [11]:
# Ask the LLM to pick a chart for a natural-language question.
question = "How does temperature vary across the basins?"

chart_response = client.messages.create(
    model=MODEL,
    max_tokens=600,
    tools=CHART_TOOLS,
    messages=[{
        "role": "user",
        "content": (
            f"Dataset schema:\n{schema_text}\n\n"
            f"User question: {question}\n\n"
            "Choose exactly one chart tool that best answers this question. "
            "Reference only columns that appear in the schema. "
            "Pick column names exactly as shown in the schema."
        ),
    }],
)

# Find the tool-use block.
tool_call = next((b for b in chart_response.content if b.type == "tool_use"), None)

print(f"Tool chosen: {tool_call.name}")
print(f"Arguments:   {dict(tool_call.input)}")

Tool chosen: create_box_plot
Arguments:   {'value_column': 'temperature_f', 'group_column': 'basin', 'title': 'Temperature Variation Across Basins'}


In [12]:
# Render the chart deterministically based on the tool call.
import plotly.express as px


def render_chart_from_tool_call(tool_name, args, df):
    if tool_name == "create_bar_chart":
        agg = args.get("aggregation", "mean")
        grouped = df.groupby(args["x_column"])[args["y_column"]].agg(agg).reset_index()
        return px.bar(grouped, x=args["x_column"], y=args["y_column"], title=args["title"])
    if tool_name == "create_line_chart":
        color = args.get("color_column") or None
        return px.line(df, x=args["x_column"], y=args["y_column"], color=color, title=args["title"])
    if tool_name == "create_scatter_chart":
        color = args.get("color_column") or None
        return px.scatter(df, x=args["x_column"], y=args["y_column"], color=color, title=args["title"])
    if tool_name == "create_histogram":
        return px.histogram(df, x=args["column"], title=args["title"])
    if tool_name == "create_box_plot":
        group = args.get("group_column") or None
        return px.box(df, x=group, y=args["value_column"], title=args["title"])
    raise ValueError(f"Unknown chart tool: {tool_name}")


figure = render_chart_from_tool_call(tool_call.name, dict(tool_call.input), df)
figure.show()

In [13]:
pip install nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Reflection 4 — On where judgment lives

Look at what just happened across the last three cells. The flow is precise:

1. **You** wrote a natural-language question
2. **The LLM** chose one chart type from a curated palette and provided structured arguments
3. **Deterministic Python code** rendered the chart based on those arguments

The boundary between LLM judgment and Python execution is exactly at the tool call. Above the boundary, the LLM exercises judgment (which chart, which columns, what title). Below the boundary, deterministic Python takes over (how the chart actually gets built). If the chart is wrong, you know exactly where to look: either the LLM picked the wrong tool, or it picked the right tool with wrong arguments. There is no fuzzy "the LLM kind of generated some code that mostly works" zone.

This is the structural reason tool use is the modern best practice for getting structured output from LLMs. It is the right level of abstraction: the model handles the interpretation, you handle the execution, and the contract between them is a typed schema.

**The takeaway:** for any structured-output task in production, tool use is the right architectural answer. The principle worth internalizing: *put the LLM's judgment behind a typed tool interface so the boundary between LLM and deterministic code is explicit and debuggable.* That principle reflects structural thinking about LLM engineering.

---

# Part 5 — AI-Narrated Captions

## Closing the loop with explanation

You have a chart. The user knows what they asked. What they don't yet have is a short, plain-English explanation of *what the chart is structured to reveal*. This is the role of the AI-narrated caption.

**The pattern:** after the chart is rendered, we make a second LLM call with the user's original question, the tool call that produced the chart, and a summary of the dataset. We ask for a short caption — 2-3 sentences — explaining what the chart shows.

**A specific instruction matters here:** we tell the LLM *not to invent numbers it cannot see*. This is the same restraint principle from Part 2. The LLM does not have access to the rendered Plotly figure; it only knows what kind of chart we built and over what columns. A caption like *"the chart shows that the Spokane basin has the highest mean temperature at 67.2°F"* would be partially fabrication — the LLM cannot read the chart, only its specification. Instead, we ask for descriptive captions like *"the chart compares mean temperature across the five basins; readers should notice which basins cluster together and which stand apart."*

In [12]:
# Generate a caption for the chart we just produced.
caption_response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": (
            f"A user asked: '{question}'\n\n"
            f"In response, we produced a {tool_call.name} with these parameters:\n"
            f"{json.dumps(dict(tool_call.input), indent=2)}\n\n"
            f"Dataset summary:\n{schema_text}\n\n"
            "Write a short, plain-English caption (2-3 sentences) explaining what this "
            "chart shows and what an analyst might notice. Do not invent numbers you "
            "cannot see; describe what the chart is structured to reveal."
        ),
    }],
)

print(caption_response.content[0].text)

This chart displays the distribution of water temperature measurements across different river basins, with each basin shown as a separate box plot. The visualization allows analysts to compare typical temperature ranges, median values, and variability between basins, making it easy to identify which basins tend to be warmer or cooler and which show more consistent versus fluctuating temperatures. Analysts can use this to spot potential outliers and understand whether certain basins experience more extreme temperature conditions than others.


### Reflection 5 — On the value of restrained captions

Read the caption. Notice what it does and does not claim. It should describe what the chart is built to compare, point toward what a viewer should look for, and avoid asserting specific numerical claims the LLM cannot verify.

This restraint is not a limitation — it is a feature. A caption that fabricates numbers is worse than no caption at all, because it gives the user false confidence. A caption that honestly describes what the chart is structured to reveal turns the chart from an artifact-to-be-interpreted into an artifact-with-orientation. The user still has to read the chart; the caption helps them know what they are reading toward.

**The principle:** the caption is restrained on purpose — it describes what the chart is built to reveal rather than claiming specific values it cannot see. This is the same principle as the data-quality flagging step: don't let the LLM produce plausible content that isn't grounded in something verifiable. Thinking about LLM failure modes structurally rather than reactively is what separates production-ready design from demos.

---

# Part 6 — Assembling the Streamlit App

## From notebook to deployable application

Each of the previous parts produced a function or pattern. The Streamlit app composes them into a single user-facing application:

| Notebook part | App function |
|---|---|
| Part 2: Dataset description | `get_dataset_description()` (cached per dataset) |
| Part 3: Data-quality flagging | `get_data_quality_concerns()` (cached per dataset) |
| Part 4: NL-to-chart | `analyze_question()` (called per user question) |
| Part 5: AI-narrated caption | (called inside `analyze_question()`) |

The full app is in `app.py` in this lab folder. Two design decisions worth understanding before you run it:

**Caching at the right boundary.** The dataset description and data-quality concerns are expensive (LLM calls) and identical across all user interactions for a given dataset. We use `@st.cache_data` with a dataset signature as the cache key, so they are computed once per dataset and reused for all subsequent interactions. The per-question analysis is *not* cached — each new question genuinely needs a fresh LLM call.

**Tool call transparency.** Below every generated chart, an expandable section shows the exact tool call that produced it. This is the debugging interface — if a chart looks wrong, the user can see exactly which tool was chosen and which arguments were passed, and form an opinion about whether the LLM made a reasonable choice. This transparency is itself part of the design: the LLM's tool call is deliberately surfaced to the user so the boundary between AI judgment and deterministic execution is auditable, not hidden.

The full source code is in `app.py`. Run it locally with:

```
pip install -r requirements.txt
export ANTHROPIC_API_KEY=sk-ant-...
streamlit run app.py
```

Or deploy to Streamlit Community Cloud — see the README for instructions and the URL you get back.

---

# Closing — What You've Built

## What is now true that wasn't true before

You have built a working LLM-augmented visual analytics application, end-to-end. It exercises every major pattern in the GenAI visualization lifecycle:

- **Integration of LLMs into the data visualization lifecycle** — end-to-end from ingestion to caption
- **Data preparation** — pandas-based loading, schema introspection, profile generation
- **Natural-language chart generation** — implemented via tool use, the current best-practice pattern
- **AI-narrated dashboards** — the caption layer demonstrates this directly
- **Interactive analytical reasoning** — each user question produces a chart and an explanation, in a loop

It also puts the underlying research argument (Minton, 2025) into operational form via the data-quality flagging step, with Talburt's definition of data quality — conformance to data specifications — explicitly anchoring the LLM prompt.

## The two design decisions worth remembering

**Tool use over code execution.** The hardest design question in a natural-language-to-chart system is how to bridge a plain-English question to an actual chart. The naive path is to have the LLM emit Python code and `exec` it — unsafe and brittle. The path this lab takes is tool use: the LLM picks from a curated palette of chart tools with structured arguments, and deterministic Python renders the chart. The boundary between LLM judgment and code execution sits exactly at the tool call, which makes the system debuggable and safe. This pattern transfers to any system that needs structured output from an LLM.

**Data quality as a first-class citizen.** The app surfaces data-quality concerns *before* analysis, not after — closing by design the awareness gap the underlying research identifies. Combined with the restrained captions (no invented numbers) and the auditable tool calls, the through-line of the whole application is the same principle: never let fluent output substitute for grounded output.

## What to do next

1. **Deploy to Streamlit Community Cloud.** A live URL is meaningfully different from a notebook on a laptop — it is the difference between *I built something* and *here is the thing I built, running, that you can also use.* The README covers the deployment steps.

2. **Run the app against a real dataset of your own.** Real data has real problems; the synthetic dataset's seeded issues are tidy by comparison. Running this against something messy will surface what works and what needs more engineering — which is itself the best next lesson.

3. **Extend it.** The README's extension ideas are ordered by leverage: a verification layer that checks column existence before rendering, multi-chart responses, conversational memory for follow-up questions, and deterministic severity scoring for the data-quality checks with the LLM explaining rather than deciding.

Good luck — and may your data always conform to its specifications.